# Synthea Explorations

This notebook explores synthetic patient data generated by [Synthea](https://github.com/synthetichealth/synthea).
Synthea outputs FHIR R4 Bundles containing Patient, Practitioner, Encounter, Condition, and other resources.

In [14]:
import os

from dicom_py_mock_server.services.synthea_utils import SyntheaUtils

SYNTHEA_OUTPUT_FOLDER = "../tests/synthea/fhir"
MAX_BUNDLES = 25


# Check if the output folder exists
if not os.path.exists(SYNTHEA_OUTPUT_FOLDER):
    print(f"Output folder '{SYNTHEA_OUTPUT_FOLDER}' does not exist.")
else:
    print(f"Output folder '{SYNTHEA_OUTPUT_FOLDER}' found.")

# Load the Synthea output data, parse them into a list of FHIR Bundle dicts, and return the list

syn = SyntheaUtils(synthea_output_folder=SYNTHEA_OUTPUT_FOLDER)

Output folder '../tests/synthea/fhir' found.


In [ ]:
from typing import List

from dicom_py_mock_server.models.fhir_models import (
    FhirOrganization,
    FhirPractitioner,
    format_human_name_to_dicom,
)

hospitals: List[FhirOrganization] = syn.get_hospital_information()
org: FhirOrganization
practitioners: List[FhirPractitioner]
p: FhirPractitioner
practitioners = syn.get_practitioner_information()
if not hospitals:
    print("Failed to get hospitals")
if not practitioners:
    print("Failed to get practitioners")
for org in hospitals:
    # print(f"{org.id}: {org.name} <- {org.partOf}")
    pass
for p in practitioners:
    name = format_human_name_to_dicom(p.name[0])
    ident = p.identifier[0].value
    print(f"{name} NPI: {ident}")

In [4]:
from typing import List

from fhir.resources.R4B.attachment import Attachment
from fhir.resources.R4B.diagnosticreport import DiagnosticReport
from fhir.resources.R4B.patient import Patient
from fhir.resources.R4B.procedure import Procedure

bundles = syn.scan_synthea_output(max_count=MAX_BUNDLES)

print(f"Loaded {len(bundles)} FHIR Bundles from Synthea output.")

for bundle in bundles:
    patient: Patient = syn.get_patient(bundle)
    dicom_name = syn.get_patient_name(patient)
    patient_id = syn.get_patient_id(patient)
    sex = syn.get_patient_sex(patient)
    birth_date = syn.get_date_time(patient.birthDate)
    print("\n====================================")
    print(f"Patient: DICOM Name: {dicom_name}, Gender: {sex}, ID: {patient_id}, Birth Date: {birth_date[0]}")
    # Imaging Studies
    imagings = syn.get_imaging_studies(bundle)
    for imaging in imagings:
        for s in imaging.series:
            modality = s.modality.display
            print("-- Imaging --")
            print(f"Modality: {modality}")
        enc_ref = imaging.encounter.reference
        encounters = syn.get_encounters(bundle=bundle, encounter_ref=enc_ref)
        for enc in encounters:
            reason = ""
            for reason_code in enc.reasonCode:
                display = reason_code.coding[0].display
            print(f"Encounter On:{enc.period.start}, Ref: {enc_ref}, Reason Code: {reason_code}")
        diags: List[DiagnosticReport] = syn.get_diagnostic_reports(bundle=bundle, encounter_ref=enc_ref)
        procedures: List[Procedure] = syn.get_procedures(bundle=bundle, encounter_ref=enc_ref)
        diag: DiagnosticReport
        proc: Procedure
        for diag in diags:
            if diag and diag.presentedForm:
                for form in diag.presentedForm:
                    match form:
                        case Attachment() as a:
                            text = a.data.decode("utf-8")
                            print(f"Diagnostic Report: {text[0:100]}")
        for proc in procedures:
            code = proc.code.text
            reason = ""
            if proc.reasonReference:
                reason = proc.reasonReference[0].display
            location = proc.location.display
            print(f"Procedure: {code}, {reason} at {location}")

Loaded 3 FHIR Bundles from Synthea output.

Patient: DICOM Name: Hettinger594^Alden634^Frankie174^Mr.^JD, Gender: M, ID: S99916806, Birth Date: 19260821
-- Imaging --
Modality: Computed Radiography
Encounter On:1995-01-21 15:46:29-08:00, Ref: urn:uuid:1bbc0b16-2965-fc1c-b8e2-01ed9b1da080, Reason Code: fhir_comments=None extension=None id=None coding=[Coding(fhir_comments=None, extension=None, id=None, code='274531002', code__ext=None, display='Abnormal findings diagnostic imaging heart+coronary circulat (finding)', display__ext=None, system='http://snomed.info/sct', system__ext=None, userSelected=None, userSelected__ext=None, version=None, version__ext=None)] text=None text__ext=None
Diagnostic Report: 
1995-01-21

# Chief Complaint
- Thirst
- Frequent Urination
- Tingling in Hands and Feet
- urinary 
Procedure: Pre-surgery testing (procedure),  at Fitchburg Outpatient Clinic
Procedure: Blood group typing (procedure),  at Fitchburg Outpatient Clinic
Procedure: Electrocardiographic proc

## 3. Direct REST API Ingestion Test

Directly test ingesting raw JSON Synthea bundles via `POST /api/v1/synthea/load` using the test assets and inspect the returned parsed bundle information.

In [17]:
import json
import os

import httpx
from fastapi.testclient import TestClient

from dicom_py_mock_server.main import app

# Re-initialize SyntheaUtils!
syn = SyntheaUtils(synthea_output_folder=SYNTHEA_OUTPUT_FOLDER)

# Prefer calling the live running dicom-py-mock-server on http://127.0.0.1:8000
# so its DICOM SCP listener (e.g. port 11112 / 11114) receives the Synthea patients/studies.
SERVER_BASE_URL = os.environ.get("GOSMART_MS_BASE_URL", "http://127.0.0.1:8000")
try:
    httpx.get(f"{SERVER_BASE_URL}/health", timeout=2.0)
    client = httpx.Client(base_url=SERVER_BASE_URL, timeout=120.0)
    print(f"Connected to live server at {SERVER_BASE_URL}")
except Exception:
    client = TestClient(app)
    print("Live server not reachable; falling back to in-process TestClient(app)")

hosp_path = os.path.join(SYNTHEA_OUTPUT_FOLDER, "hospitalInformation1791996124090.json")
prac_path = os.path.join(SYNTHEA_OUTPUT_FOLDER, "practitionerInformation1791996124090.json")

with open(hosp_path, "r", encoding="utf-8") as f:
    hosp_json = json.load(f)
with open(prac_path, "r", encoding="utf-8") as f:
    prac_json = json.load(f)


# Step 1: Load hospital and practitioner resources into the mock server
init_resp = client.post("/api/v1/synthea/load", json=[hosp_json, prac_json])
print(f"Loaded prerequisites (hospitals & practitioners): status {init_resp.status_code}")

# Step 2: Load the patient bundles
raw_bundles = []
for file_name in sorted(os.listdir(syn.synthea_output_folder)):
    if "hospital" in file_name or "practitioner" in file_name or not file_name.endswith(".json"):
        continue
    with open(os.path.join(SYNTHEA_OUTPUT_FOLDER, file_name), "r", encoding="utf-8") as f:
        raw_bundles.append(json.load(f))

load_resp = client.post("/api/v1/synthea/load", json=raw_bundles)
print(f"POST /api/v1/synthea/load patient bundle: status {load_resp.status_code}")
returned_bundles = load_resp.json()

# Step 3: Display essential bundle information returned
print(f"Returned {len(returned_bundles)} parsed FHIR Bundle(s):")

for idx, bundle in enumerate(returned_bundles):
    bundle_id = bundle.get("id")
    entries = bundle.get("entry", [])
    entry_types = [e.get("resource", {}).get("resourceType") for e in entries if "resource" in e]
    patient_resource = next(
        (e["resource"] for e in entries if e.get("resource", {}).get("resourceType") == "Patient"),
        None,
    )
    studies = [e["resource"] for e in entries if e.get("resource", {}).get("resourceType") == "ImagingStudy"]

    print(f"\nBundle #{idx + 1}: ID={bundle_id}")
    print(f"  Total Entries: {len(entries)}")
    print(f"  Resource Types: {set(entry_types)}")
    if patient_resource:
        print(f"  Patient ID: {patient_resource.get('id')}")
        if patient_resource.get("name"):
            print(f"  Patient Name: {patient_resource['name'][0]}")
    print(f"  ImagingStudies: {len(studies)}")
    for s_idx, study in enumerate(studies):
        modalities = [s.get("modality", {}).get("code") for s in study.get("series", [])]
        print(f"    Study #{s_idx + 1}: ID={study.get('id')}, Modalities={modalities}")

2026-10-19T19:58:52.185496Z [info     ] HTTP Request: GET http://127.0.0.1:8000/health "HTTP/1.1 200 OK" [httpx] func_name=_send_single_request lineno=1025 method_name=_send_single_request
Connected to live server at http://127.0.0.1:8000
2026-10-19T19:58:52.480479Z [info     ] HTTP Request: POST http://127.0.0.1:8000/api/v1/synthea/load "HTTP/1.1 200 OK" [httpx] func_name=_send_single_request lineno=1025 method_name=_send_single_request
Loaded prerequisites (hospitals & practitioners): status 200
2026-10-19T19:59:01.942083Z [info     ] HTTP Request: POST http://127.0.0.1:8000/api/v1/synthea/load "HTTP/1.1 200 OK" [httpx] func_name=_send_single_request lineno=1025 method_name=_send_single_request
POST /api/v1/synthea/load patient bundle: status 200
Returned 3 parsed FHIR Bundle(s):

Bundle #1: ID=None
  Total Entries: 8490
  Resource Types: {'DocumentReference', 'ImagingStudy', 'Patient', 'Condition', 'Claim', 'Encounter', 'CarePlan', 'Device', 'Observation', 'Medication', 'Immunizatio